# Kaggle: `dpo_from_sft` -- warm-started DPO training -> evaluate

Runs, in order, against `Qwen/Qwen2.5-3B-Instruct`, warm-started from the already-trained `sft_qlora` adapter:

1. **DPO training** (`src/train.train_dpo`, warm-started via `--adapter` from `sft_qlora` -- continues those same LoRA weights rather than a fresh LoRA, per `config.yaml`'s `runs: dpo_from_sft: warm_start: sft_qlora`). Hyperparameters read from `config.yaml`'s `training.dpo` section, same "don't re-type hyperparameters into the notebook" rule as the SFT notebook.
2. **`dpo_from_sft` eval** on the *same* 803-row test split used for `baseline`/`sft_qlora` (not the DPO-format test file -- that's only for the DPO trainer's own held-out loss, this is for the apples-to-apples rubric comparison).
3. Score with the deterministic rubric and append to `results/summary.csv`.

Reuses every fix already found for the SFT notebook (see `LOG.md` 2026-08-18): single-GPU device pin, gradient checkpointing, capability-aware bf16/fp16, Unsloth with a post-install CUDA-survived check. `--use-unsloth` is on by default here from the start -- no need to rediscover the same 117-125s/step problem a second time.

**Important, more than usual**: `train_dpo.py` is *less* locally verified than `train_sft.py` was. `trl.DPOTrainer` fails to import at all on the local dev machine (needs `torch.distributed.fsdp.FSDPModule`, added in a PyTorch release newer than the locally-pinned `2.5.1`) -- only the dataset-loading logic could be checked locally, not the trainer construction or training loop. Kaggle's newer PyTorch is expected to have this, but the dry-run cell below matters even more here than it did for SFT -- watch it closely. See `LOG.md` 2026-08-19.

**Before running:**
1. Zip `src/` (contents) + `config.yaml` together as `src.zip`, same as before -- this includes the new `train_dpo.py`.
2. Upload that zip, plus `data/splits/dpo_train.jsonl`, `data/splits/dpo_valid.jsonl`, `data/splits/sft_test.jsonl` (for eval, same test set as before), and the downloaded `adapters/sft_qlora/` directory (for the warm-start), as a Kaggle Dataset.
3. Attach the dataset, single T4 accelerator, Internet on.
4. Run all cells. A small dry run happens automatically before the full run -- check its timing (should be in the same ~7-8s/step ballpark as the SFT run) before letting the full run proceed.

**Output:** `/kaggle/working/adapters/dpo_from_sft/` (download to `adapters/dpo_from_sft/` locally), `/kaggle/working/results/` (`dpo_from_sft_gen.jsonl`, `dpo_from_sft_scored.jsonl`, `summary.csv` with just the `dpo_from_sft` row -- merge into local `results/summary.csv` same as the `sft_qlora` row was).

In [ ]:
!pip install -q -U "trl==1.10.0" "peft==0.20.0" "transformers==5.15.0" "bitsandbytes==0.50.1" "accelerate==1.14.0" pyyaml
# peft 0.20.0 requires torchao>=0.16.0 for an internal LoRA-dispatch check;
# Kaggle's base image ships torchao==0.10.0. Only bites when loading a PEFT
# adapter onto a full-precision (non-4-bit) base model -- not the case in
# this notebook's own eval calls, but uninstalled defensively anyway since
# this project doesn't use torchao at all and it cost a full 3h15m training
# run + eval crash to find in kaggle_sft_lora_fp.ipynb. See LOG.md 2026-08-19.
!pip uninstall -y -q torchao
import torch
print("CUDA available:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## Unsloth (accelerated QLoRA path)

`--no-deps`: a bare `pip install unsloth` silently upgraded `torch` and broke CUDA both on the local dev machine (2026-08-18) *and* on this notebook's first Kaggle run (2026-08-19) -- confirmed non-deterministic across sessions, not a one-off. `--no-deps` prevents pip from touching the already-working `torch`/`transformers`/etc.; `unsloth_zoo` needs the same flag since it's unsloth's own runtime dependency. The CUDA-survived assertion stays as a safety net in case something else still slips through.

In [ ]:
!pip install -q --no-deps unsloth unsloth_zoo
import torch
print("CUDA available after unsloth install:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
assert torch.cuda.is_available(), (
    "CUDA broke after installing unsloth even with --no-deps -- see LOG.md 2026-08-19. Do not "
    "proceed with --use-unsloth training if this assertion fails; fall back to USE_UNSLOTH=False "
    "instead, or inspect `pip list` for what changed."
)

In [ ]:
import os, sys, zipfile

def find_repo(root="/kaggle/input"):
    """Same marker-based search as the SFT notebook -- locates src/ and
    config.yaml independently by content marker rather than assuming a
    fixed relative layout. See LOG.md 2026-08-17/18."""
    repo_dir = None
    config_path = None
    zip_path = None
    for r, dirs, files in os.walk(root):
        if "build_irac.py" in files and os.path.basename(r) == "data":
            src_dir = os.path.dirname(r)
            if os.path.basename(src_dir) == "src":
                repo_dir = os.path.dirname(src_dir)
        if config_path is None and "config.yaml" in files:
            config_path = os.path.join(r, "config.yaml")
        if "src.zip" in files:
            zip_path = os.path.join(r, "src.zip")
    return repo_dir, config_path, zip_path

repo_dir, config_path, zip_path = find_repo()

if repo_dir is None and zip_path:
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("/kaggle/working/repo")
    repo_dir, config_path, _ = find_repo("/kaggle/working/repo")
    print("Extracted", zip_path)

if repo_dir is None or config_path is None:
    raise FileNotFoundError(
        f"repo_dir={repo_dir}, config_path={config_path} -- could not find both "
        "src/data/build_irac.py and config.yaml under /kaggle/input (in any "
        "nesting). Check the dataset is attached. If just attached/updated, "
        "try Restart & Run All."
    )

REPO_DIR = repo_dir
sys.path.insert(0, REPO_DIR)
print("REPO_DIR =", REPO_DIR)
print("config.yaml at", config_path)

import yaml
with open(config_path) as f:
    cfg = yaml.safe_load(f)

MODEL_ID = cfg["model"]["candidates"][cfg["model"]["active"]]["hf_id"]
print("Active model:", MODEL_ID)

In [ ]:
import os

def find_data_file(name, root="/kaggle/input"):
    for r, dirs, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    raise FileNotFoundError(f"{name} not found under {root} -- check it was included in the uploaded dataset.")

def find_adapter_dir(root="/kaggle/input"):
    """Content-marker search, not a name match -- just find wherever
    adapter_config.json + adapter_model.safetensors landed, regardless of
    what folder (if any) they're nested under. Doesn't assume a folder
    literally named 'sft_qlora' exists, since the upload might have been
    zipped as loose files rather than a wrapped folder -- same class of
    nesting surprise as everything else in this project.

    Excludes 'dryrun' paths and intermediate epoch checkpoints
    explicitly and requires exactly one survivor: a raw Kaggle output
    upload can contain the dry-run checkpoint, the real adapter's own
    epoch checkpoints (save_strategy="epoch" writes a full adapter to
    output_dir/checkpoint-N/ at every epoch on top of the final state at
    the top level), or both, all matching the same content marker.
    os.walk's traversal order isn't guaranteed alphabetical on Kaggle's
    filesystem, so returning "the first match" could silently grab a
    stale/untrained one with no error. See LOG.md 2026-08-20."""
    candidates = []
    for r, dirs, files in os.walk(root):
        if "adapter_config.json" in files and "adapter_model.safetensors" in files:
            candidates.append(r)
    real = [c for c in candidates if "dryrun" not in c.lower() and "checkpoint-" not in c.lower()]
    if len(real) == 1:
        return real[0]
    raise FileNotFoundError(
        f"Expected exactly one non-dryrun adapter directory under {root}, found {len(real)}: "
        f"{real}. All adapter-like directories found (including dry-run): {candidates}. "
        "Resolve the ambiguity (or absence) before proceeding rather than guessing."
    )

DPO_TRAIN_FILE = find_data_file("dpo_train.jsonl")
DPO_VALID_FILE = find_data_file("dpo_valid.jsonl")
TEST_FILE = find_data_file("sft_test.jsonl")
SFT_QLORA_ADAPTER = find_adapter_dir()
print(DPO_TRAIN_FILE, DPO_VALID_FILE, TEST_FILE, SFT_QLORA_ADAPTER, sep="\n")

RESULTS_DIR = "/kaggle/working/results"
ADAPTERS_DIR = "/kaggle/working/adapters"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(ADAPTERS_DIR, exist_ok=True)
SUMMARY_CSV = os.path.join(RESULTS_DIR, "summary.csv")

from src.eval.generate import run as generate_run
from src.eval.score import score_file, append_summary_row

GEN_BATCH_SIZE = 16

## DPO training (`dpo_from_sft`, warm-started)

Hyperparameters come from `config.yaml`'s `peft` and `training.dpo` sections. `--use-unsloth` is on by default. A short `--max-steps` dry run proves the path works (model loads, warm-start attaches correctly, a step actually runs) before committing to the full run.

In [ ]:
peft_cfg = cfg["peft"]
dpo_cfg = cfg["training"]["dpo"]
max_seq_length = cfg["training"]["max_seq_length"]

DPO_FROM_SFT_DIR = os.path.join(ADAPTERS_DIR, "dpo_from_sft")

USE_UNSLOTH = True

def train_dpo_args(output_dir, max_steps=None, num_epochs=None):
    args = [
        "--model", MODEL_ID,
        "--adapter", SFT_QLORA_ADAPTER,
        "--train-file", DPO_TRAIN_FILE,
        "--eval-file", DPO_VALID_FILE,
        "--output-dir", output_dir,
        "--load-in-4bit",
        "--lora-r", str(peft_cfg["lora_r"]),
        "--lora-alpha", str(peft_cfg["lora_alpha"]),
        "--lora-dropout", str(peft_cfg["lora_dropout"]),
        "--target-modules", *peft_cfg["target_modules"],
        "--max-seq-length", str(max_seq_length),
        "--beta", str(dpo_cfg["beta"]),
        "--per-device-batch-size", str(dpo_cfg["per_device_batch_size"]),
        "--gradient-accumulation-steps", str(dpo_cfg["gradient_accumulation_steps"]),
        "--learning-rate", str(dpo_cfg["learning_rate"]),
    ]
    if USE_UNSLOTH:
        args += ["--use-unsloth"]
    if max_steps is not None:
        args += ["--max-steps", str(max_steps)]
    else:
        args += ["--num-epochs", str(num_epochs)]
    return args

print(train_dpo_args(DPO_FROM_SFT_DIR, num_epochs=dpo_cfg["epochs"]))

In [ ]:
import subprocess, sys

env = os.environ.copy()
env["PYTHONPATH"] = REPO_DIR
env["PYTHONUNBUFFERED"] = "1"

dryrun_dir = os.path.join(ADAPTERS_DIR, "dpo_from_sft_dryrun")
cmd = [sys.executable, "-m", "src.train.train_dpo"] + train_dpo_args(dryrun_dir, max_steps=5)
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=env)

In [ ]:
# Full DPO training run. Check the dry-run cell's per-step timing above and
# multiply by the expected total steps (5,987 rows / effective batch *
# epochs -- same math as the SFT run) for a real ETA before running this.
# The SFT run landed around ~7-8s/step with this same stack (single-GPU
# pin + Unsloth + capability-aware bf16); this should land in the same
# ballpark. If the dry run looks anywhere near the pre-fix 117-125s/step,
# stop and report the timing rather than let this run for hours. See
# LOG.md 2026-08-18/19.
cmd = [sys.executable, "-m", "src.train.train_dpo"] + train_dpo_args(DPO_FROM_SFT_DIR, num_epochs=dpo_cfg["epochs"])
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=env)

## `dpo_from_sft` eval

Same test split, same generation script, `dpo_from_sft` adapter attached instead of `sft_qlora`.

In [ ]:
DPO_FROM_SFT_GEN = os.path.join(RESULTS_DIR, "dpo_from_sft_gen.jsonl")

generate_run(
    input_path=TEST_FILE,
    output_path=DPO_FROM_SFT_GEN,
    model_id=MODEL_ID,
    adapter_path=DPO_FROM_SFT_DIR,
    load_in_4bit=True,
    batch_size=GEN_BATCH_SIZE,
    max_new_tokens=350,
    limit=None,
)

In [ ]:
DPO_FROM_SFT_SCORED = os.path.join(RESULTS_DIR, "dpo_from_sft_scored.jsonl")
dpo_from_sft_summary = score_file(DPO_FROM_SFT_GEN, DPO_FROM_SFT_SCORED)
append_summary_row("dpo_from_sft", dpo_from_sft_summary, SUMMARY_CSV)
print("dpo_from_sft:", dpo_from_sft_summary)

In [ ]:
import pandas as pd
df = pd.read_csv(SUMMARY_CSV)
print(df.to_string(index=False))